# Problema 1: Análise de Custos de Serviços em Nuvem

Projeto Métodos Numéricos 2ºGQ. As funções vêm do pacote `custos_nuvem` (instale com `pip install -e .` na raiz do repositório).

In [1]:
import numpy as np
import pandas as pd
from custos_nuvem import (
    carregar_csv, vetor_previsao, montar_sistema, custos_unitarios,
    verificar_totais, custo_previsto, melhor_reducao,
    eliminacao_gauss, resolver_lu,
)

## Dados

Ajuste os nomes das colunas conforme os CSVs (e `sep`/`decimal` se necessário).

In [2]:
consumo = carregar_csv('../data/consumo_servicos.csv')
previsao = carregar_csv('../data/previsao.csv')
display(consumo)
display(previsao)

,mes,cpu_horas,armazenamento_tb,banco_horas,transferencia_gb,funcoes_mil,custo_total
0,1,100.0,20.0,80.0,200.0,150.0,1255.0
1,2,120.0,25.0,90.0,180.0,200.0,1479.0
2,3,150.0,30.0,100.0,250.0,180.0,1695.0
3,4,130.0,35.0,120.0,220.0,250.0,1881.0
4,5,180.0,40.0,110.0,300.0,220.0,2060.0


,mes,cpu_horas,armazenamento_tb,banco_horas,transferencia_gb,funcoes_mil
0,6,200.0,45.0,130.0,320.0,280.0


In [3]:
# Colunas dos CSVs do enunciado
SERVICOS = ['cpu_horas', 'armazenamento_tb', 'banco_horas', 'transferencia_gb', 'funcoes_mil']
COLUNA_TOTAL = 'custo_total'

## (a) Custo unitário de cada serviço

Sistema linear 5×5 `A·x = b`: `A` = quantidades consumidas, `b` = total pago por mês.

In [4]:
A, b = montar_sistema(consumo, SERVICOS, COLUNA_TOTAL)
x = custos_unitarios(A, b)              # eliminação de Gauss com pivoteamento
x_lu = resolver_lu(A, b)                # conferência com LU
pd.DataFrame({'gauss': x, 'lu': x_lu}, index=SERVICOS)

,gauss,lu
cpu_horas,2.5,2.5
armazenamento_tb,15.0,15.0
banco_horas,4.0,4.0
transferencia_gb,0.8,0.8
funcoes_mil,1.5,1.5


## (b) Verificação dos totais de faturamento

In [5]:
verificar_totais(A, x, b)

,mes,total_calculado,total_original,erro_absoluto,erro_relativo
0,1,1255.0,1255.0,0.0,0.0
1,2,1479.0,1479.0,0.0,0.0
2,3,1695.0,1695.0,0.0,0.0
3,4,1881.0,1881.0,0.0,0.0
4,5,2060.0,2060.0,0.0,0.0


**Interpretação:** _(Guilherme: comentar erros, arredondamento e condicionamento.)_

## (c) Custo total estimado do próximo mês

In [6]:
q = vetor_previsao(previsao, SERVICOS)
custo_previsto(q, x)

2371.0

## (d) Redução de 20% em um serviço

In [7]:
r = melhor_reducao(q, x, SERVICOS, reducao=0.20, meses=12)
print('Serviço escolhido:', r['servico'])
print('Economia mensal:', r['economia_mensal'])
print('Novo custo total:', r['novo_total'])
print('Economia em 12 meses:', r['economia_acumulada'])
r['economias']

Serviço escolhido: armazenamento_tb
Economia mensal: 135.00000000000003
Novo custo total: 2236.0
Economia em 12 meses: 1620.0000000000005


cpu_horas           100.0
armazenamento_tb    135.0
banco_horas         104.0
transferencia_gb     51.2
funcoes_mil          84.0
dtype: float64

## Conclusões e limitações

_Preencher._